# FinFET 2nm Node — Boron Source/Drain Diffusion Case Study

**Author:** Froylan González | Nanotechnology Engineering  
**Tool:** NanoDopant-Sim — TCAD Dopant Diffusion Simulator

---

## Context: Why Dopant Profiles Matter at 2nm

At the 2nm node (GAAFET/FinFET architectures), the source and drain extensions are separated
from the channel by only **4–6 nm**. Boron (p-type) is the standard dopant for PMOS source/drain
in Si and SiGe fins. Precise control of the junction depth x_j and the dopant gradient at the
metallurgical junction is critical for:

- **Short-channel effect (SCE) suppression** — abrupt junctions reduce drain-induced barrier lowering (DIBL)
- **Series resistance minimization** — high peak concentration near surface reduces contact resistance
- **Thermal budget compliance** — modern flows allow < 1000°C spike anneal to limit dopant redistribution

The diffusion profile is governed by **Fick's Second Law** with the **erfc** boundary condition
(constant surface source, representative of ion implant + drive-in):

$$C(x,t) = C_s \cdot \text{erfc}\!\left(\frac{x}{2\sqrt{Dt}}\right)$$

where $D = D_0 e^{-E_a / k_B T}$ (Arrhenius).

In [ ]:
import sys
sys.path.insert(0, '..')  # allow imports from project root

import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio
pio.renderers.default = 'notebook'

from physics_engine import DiffusionParams, solve_analytic, compute_diffusivity
from visualizer import plot_profile, plot_heatmap, plot_animation_html, plot_animation_gif

print('NanoDopant-Sim loaded successfully')

## Simulation Setup — Boron in Si at 1000°C

Parameters representative of a PMOS FinFET source/drain anneal:

| Parameter | Value | Notes |
|-----------|-------|-------|
| Dopant | Boron (B) | p-type, D₀ = 0.76 cm²/s, Eₐ = 3.46 eV |
| Temperature | 1000 °C | Conservative RTP anneal |
| Time | 3600 s | 1-hour drive-in |
| Profile | erfc | Infinite-source (continuous surface supply) |
| Surface conc. | 1×10²⁰ cm⁻³ | Solid solubility limit for B in Si |

In [ ]:
params = DiffusionParams(
    dopant='B',
    temperature_C=1000.0,
    time_s=3600.0,
    profile='erfc',
    depth_um=0.5,
    surface_conc=1e20,
    n_points=500,
)

result = solve_analytic(params)

print(f'D_eff (B at 1000°C) = {result.D_eff:.4e} cm²/s')
print(f'Diffusion length √(Dt) = {np.sqrt(result.D_eff * params.time_s)*1e4:.4f} µm')
print(f'Junction depth x_j     = {result.junction_depth_um:.4f} µm')
print(f'(where C drops to 1×10¹⁶ cm⁻³ background doping)')

## Visualization 1 — Static Concentration Profile

In [ ]:
plot_profile(result, 'boron_profile.png')

from IPython.display import Image
Image('boron_profile.png', width=600)

## Visualization 2 — Interactive Heatmap (Depth × Time)

In [ ]:
plot_heatmap(result, 'boron_heatmap.html')

from IPython.display import IFrame
IFrame('boron_heatmap.html', width=750, height=450)

## Visualization 3 — Animated Profile Evolution

In [ ]:
plot_animation_gif(result, 'boron_evolution.gif')

from IPython.display import Image
Image('boron_evolution.gif', width=600)

## Results Analysis — Comparison with ITRS/Industry Targets

For a 2nm-class FinFET PMOS, the target junction depth is **x_j < 10 nm = 0.010 µm**
to suppress short-channel effects. A 1-hour anneal at 1000°C produces a junction
significantly deeper than this target — demonstrating why modern processes use:

- **Spike rapid thermal processing (RTP)** at 1050–1100°C for < 1 second
- **Flash/laser annealing** for sub-millisecond thermal budgets
- **Low-temperature millisecond anneal** after high-dose implant

In [ ]:
scenarios = [
    ('RTP spike, 1050°C, 1s',   1050, 1),
    ('RTP spike, 1050°C, 10s',  1050, 10),
    ('Drive-in, 1000°C, 3600s', 1000, 3600),
]

fig, ax = plt.subplots(figsize=(9, 5))

for label, temp, time in scenarios:
    p = DiffusionParams('B', temp, time, 'erfc', 0.5, 1e20, n_points=500)
    r = solve_analytic(p)
    mask = r.concentration > 0
    ax.semilogy(r.depth[mask], r.concentration[mask], label=f'{label}  (x_j={r.junction_depth_um:.4f} µm)')

ax.axvline(0.010, color='k', linestyle=':', linewidth=1.5, label='2nm node target x_j = 10 nm')
ax.axhline(1e16, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='Background doping 10¹⁶ cm⁻³')
ax.set_xlabel('Depth (µm)', fontsize=12)
ax.set_ylabel('Concentration (cm⁻³)', fontsize=12)
ax.set_title('Boron in Si — Anneal Condition Comparison', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(0, 0.15)
plt.tight_layout()
plt.savefig('anneal_comparison.png', dpi=150)
plt.show()
print('Saved anneal_comparison.png')

## Connection to Experimental Validation — XRD, SIMS, TEM

Simulation outputs like this are validated experimentally using:

| Technique | What it measures | Connection to model |
|-----------|-----------------|--------------------|
| **SIMS** (Secondary Ion Mass Spectrometry) | C(x) concentration profile directly | Direct ground truth for erfc/Gaussian shape |
| **XRD** (X-ray Diffraction) | Lattice strain from dopant-induced distortion | Verifies peak dopant concentration via Vegard's law |
| **TEM/STEM** | Physical junction abruptness, crystal quality | Validates absence of extended defects post-anneal |
| **4-point probe** | Sheet resistance ρ_s | Integrates C(x)·μ(x) — corroborates D_eff |

As a Nanotechnology graduate with XRD and TEM characterization experience, the bridge
between this simulation and measurement is direct: SIMS profiles extracted from test
wafers are fitted against the erfc solution to extract the experimental D_eff,
which is then compared to the Arrhenius prediction. Deviations reveal oxidation-enhanced
diffusion (OED), clustering, or non-equilibrium effects not captured by this first-order model —
motivating extensions like those implemented in production TCAD tools (Sentaurus, Silvaco).